## 1. Nhập thư viện và đọc dữ liệu

In [81]:
# Nhập thư viện
import pandas as pd
import numpy as np
import math

In [82]:
# Đọc dữ liệu
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/letter-recognition/letter-recognition.data"
cols = ['letter', 'x-box', 'y-box', 'width', 'high', 'onpix', 'x-bar', 'y-bar',
        'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']
data = pd.read_csv(url, header=None, names=cols)


In [83]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   letter  20000 non-null  object
 1   x-box   20000 non-null  int64 
 2   y-box   20000 non-null  int64 
 3   width   20000 non-null  int64 
 4   high    20000 non-null  int64 
 5   onpix   20000 non-null  int64 
 6   x-bar   20000 non-null  int64 
 7   y-bar   20000 non-null  int64 
 8   x2bar   20000 non-null  int64 
 9   y2bar   20000 non-null  int64 
 10  xybar   20000 non-null  int64 
 11  x2ybr   20000 non-null  int64 
 12  xy2br   20000 non-null  int64 
 13  x-ege   20000 non-null  int64 
 14  xegvy   20000 non-null  int64 
 15  y-ege   20000 non-null  int64 
 16  yegvx   20000 non-null  int64 
dtypes: int64(16), object(1)
memory usage: 2.6+ MB


In [84]:
print(data.head())

  letter  x-box  y-box  width  high  onpix  x-bar  y-bar  x2bar  y2bar  xybar  \
0      T      2      8      3     5      1      8     13      0      6      6   
1      I      5     12      3     7      2     10      5      5      4     13   
2      D      4     11      6     8      6     10      6      2      6     10   
3      N      7     11      6     6      3      5      9      4      6      4   
4      G      2      1      3     1      1      8      6      6      6      6   

   x2ybr  xy2br  x-ege  xegvy  y-ege  yegvx  
0     10      8      0      8      0      8  
1      3      9      2      8      4     10  
2      3      7      3      7      3      9  
3      4     10      6     10      2      8  
4      5      9      1      7      5     10  


## 2. Chuyển kí tự sang dạng số

In [85]:
# Chuyển nhãn sang số (A=0, B=1, ..., Z=25)
letters = sorted(data['letter'].unique())
label_to_int = {l:i for i, l in enumerate(letters)} # biến nhãn chữ thành số 0..25
int_to_label = {i:l for l, i in label_to_int.items()} # đảo ngược dùng khi in kết quả.
data['label'] = data['letter'].map(label_to_int)
data = data.drop(columns=['letter'])

## 3. Chia dữ liệu train/test

In [86]:
# Chia dữ liệu train/test 70/30
np.random.seed(42)
indices = np.random.permutation(len(data))
train_size = int(0.7 * len(data))
train_idx, test_idx = indices[:train_size], indices[train_size:]

train = data.iloc[train_idx]
test  = data.iloc[test_idx]

X_train, y_train = train.drop(columns=['label']).values, train['label'].values
X_test, y_test   = test.drop(columns=['label']).values, test['label'].values

In [87]:
print(f"Train shape: {len(X_train)} x {len(X_train[0])}")
print(f"Test shape:  {len(X_test)} x {len(X_test[0])}")

Train shape: 14000 x 16
Test shape:  6000 x 16


## 4. Huấn luyện Naive Bayes

## 4.1 Tính trung bình, phương sai và prior

In [88]:
# Huấn luyện Naive Bayes
classes = np.unique(y_train)

# Tính trung bình, phương sai và prior cho mỗi lớp
mean = {}
var  = {}
prior = {}

for c in classes:
    X_c = X_train[y_train == c]
    mean[c] = X_c.mean(axis=0)
    var[c]  = X_c.var(axis=0) + 1e-6   # tránh chia cho 0
    prior[c] = len(X_c) / len(X_train)

## 4.2 Dự đoán

In [89]:
# Dự đoán
# Trả về đúng giá trị log-likelihood
def likelihood(x, mean, var):
    # Tính log xác suất để tránh tràn số
    return -0.5 * np.sum(np.log(2.0 * np.pi * var)) - 0.5 * np.sum(((x - mean)**2) / var)

def predict(x):
    posteriors = []
    for c in classes:
        log_prior = math.log(prior[c])
        log_likelihood = likelihood(x, mean[c], var[c])
        posterior = log_prior + log_likelihood
        posteriors.append(posterior)
    return classes[np.argmax(posteriors)]

### 4.3 Kết quả

In [90]:
# Kiểm thử mô hình
predictions = [predict(x) for x in X_test]
accuracy = np.mean(np.array(predictions) == y_test)
print(f"Độ chính xác: {accuracy * 100:.2f}%")

Độ chính xác: 64.48%


In [91]:
#  Hiển thị vài kết quả mẫu
for i in range(15):
    true_label = int_to_label[y_test[i]]
    pred_label = int_to_label[predictions[i]]
    print(f"Mẫu {i+1}: Thật = {true_label}, Dự đoán = {pred_label}")

Mẫu 1: Thật = P, Dự đoán = P
Mẫu 2: Thật = Y, Dự đoán = W
Mẫu 3: Thật = H, Dự đoán = Q
Mẫu 4: Thật = M, Dự đoán = M
Mẫu 5: Thật = X, Dự đoán = X
Mẫu 6: Thật = X, Dự đoán = B
Mẫu 7: Thật = I, Dự đoán = I
Mẫu 8: Thật = T, Dự đoán = T
Mẫu 9: Thật = A, Dự đoán = A
Mẫu 10: Thật = V, Dự đoán = V
Mẫu 11: Thật = S, Dự đoán = I
Mẫu 12: Thật = D, Dự đoán = D
Mẫu 13: Thật = K, Dự đoán = C
Mẫu 14: Thật = C, Dự đoán = C
Mẫu 15: Thật = E, Dự đoán = E
